# Interested in viewing the bathymetry or mesh that was used in the model to create the CORA data? This notebook allows users to create a rasterized plot of the topobathy at the CORA model nodes and overlay the model mesh.

*Parts of the code in this notebook were guided by **Rudiger, P. (2023). Bay Trimesh. https://github.com/holoviz/datashader/blob/f23de596f9adcb8188d48e6b163c36c913cd9912/examples/user_guide/6_Trimesh.ipynb#L11***

In [ ]:
import os

from scipy.spatial import cKDTree
import numpy as np
import requests
import matplotlib.pyplot as plt
import pandas as pd
import dask
import intake
import xarray as xr
import holoviews as hv
import datashader as ds
import datashader.transfer_functions as tf
import datashader.utils as du
import geoviews as gv
from holoviews import opts
import holoviews.operation.datashader as dshade
from holoviews.operation.datashader import datashade, shade, rasterize
import cmocean
hv.extension('bokeh')

**Access the data on the NODD and initialize the available CORA datasets.** 

*This accesses a .yml file located on the NODD that shows which CORA output files are available to import.*

In [ ]:

catalog = intake.open_catalog("s3://noaa-nos-cora-pds/CORA_V1.1_intake.yml",storage_options={'anon':True})
list(catalog)


**CORA-V1.1-fort.63: Hourly water levels <br>
CORA-V1.1-swan_DIR.63: Hourly mean wave direction <br>
CORA-V1.1-swan_TPS.63: Hourly peak wave periods <br>
CORA-V1.1-swan_HS.63: Hourly significant wave heights <br>
CORA-V1.1-Grid: Hourly water levels interpolated from model nodes to uniform 500-meter resolution grid <br>
All datasets denoted as timeseries are optimized for pulling long time series (greater than a few days) <br>
For up to a few days of data, use the regular dataset (not labeled timeseries)**

*Now, create an xarray dataset for the CORA data that you would like to use.*

In [ ]:
cora = catalog["CORA-V1.1-fort.63"].to_dask()


In [ ]:
def filter_mesh_to_region(verts_df, tris_df, bounds):
    """
    More mesh filtering using vectorized operations
    """
    # Create mask for vertices in region
    mask = (
        (verts_df['x'] >= bounds['lon_min']) &
        (verts_df['x'] <= bounds['lon_max']) &
        (verts_df['y'] >= bounds['lat_min']) &
        (verts_df['y'] <= bounds['lat_max'])
    )

    # Get vertices in region
    region_verts = verts_df[mask].copy().reset_index()

    # Create efficient index mapping using numpy
    old_indices = region_verts['index'].values
    new_indices = np.arange(len(region_verts))
    index_map = dict(zip(old_indices, new_indices))

    # Only keep triangles where ALL vertices are in the region
    valid_mask = np.isin(tris_df.values, old_indices).all(axis=1)
    region_tris = tris_df[valid_mask].copy()

    # Remap triangle indices
    for col in ['v0', 'v1', 'v2']:
        region_tris[col] = region_tris[col].map(index_map)

    # Remove the original index column
    region_verts = region_verts.drop('index', axis=1).reset_index(drop=True)
    region_tris = region_tris.reset_index(drop=True)

    return region_verts, region_tris

In [ ]:
# Create vertices and triangles from the full dataset
v = np.vstack((cora.x, cora.y, cora.depth)).T
verts_original = pd.DataFrame(v, columns=['x', 'y', 'z'])
tris_original = pd.DataFrame(cora['element'].values.astype('int')-1, columns=['v0','v1','v2'])

# Define Texas bounding box
texas_bounds = {'lon_min': -98.0, 'lon_max': -93.0, 'lat_min': 25.0, 'lat_max': 31.0}

# Use the efficient filtering function
verts, tris = filter_mesh_to_region(verts_original, tris_original, texas_bounds)

print(f"Original vertices: {len(verts_original)}, Texas coast vertices: {len(verts)}")
print(f"Original triangles: {len(tris_original)}, Texas coast triangles: {len(tris)}")

points = gv.operation.project_points(gv.Points(verts, vdims=['z']))

In [ ]:
base_url = 'https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations.json'
params = {
    'type': 'waterlevels',
    # 'type': 'historicwl',
    'units': 'metric'
}
print(f'base_url: {base_url}, parameters: {params}')
response = requests.get(base_url, params=params)
content = response.json()

stations = content['stations']
stations_df = pd.DataFrame(stations)

# Include station name along with id, lat, lng
stations_df = stations_df[['id','name','lat','lng','state']]

# limit to Texas stations
stations_df = stations_df[stations_df['state'] == 'TX']

# Filter for Texas stations
texas_stations = stations_df[stations_df['state'] == 'TX'].copy()

print(f"Found {len(texas_stations)} stations:")
texas_stations = texas_stations.reset_index(drop=True)
display(texas_stations)

In [ ]:
# Create station points for overlay
station_points = hv.Points(
    texas_stations,
    kdims=['lng', 'lat'],
    vdims=['name', 'id']
).opts(
    size=8,
    color='red',
    marker='circle',
    line_color='white',
    line_width=0.5,
    tools=['hover'],
    alpha=0.8
)



opts.defaults(
    opts.Image(width=1000, height=700),
    opts.RGB(width=1000, height=700))

# Create wireframe for mesh overlay
# the alpha value for the mesh lines is from 0 - 254
wireframe = datashade(hv.TriMesh((tris,verts)).edgepaths, alpha=128)

# Create the trimesh with depth values
trimesh = hv.TriMesh((tris, hv.Points(verts, vdims='z')))

dmesh = dshade.rasterize(trimesh).opts(cmap=cmocean.cm.deep, colorbar=True)

# Overlay the bathymetry, wireframe, and station points
dmesh * wireframe * station_points


In [ ]:
import numpy as np
import pandas as pd

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees).
    Returns distance in kilometers.
    """
    # Convert to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    # Radius of Earth in kilometers
    r = 6371.0

    return c * r

def find_surrounding_triangles(station_lat, station_lng, verts, tris, radius_km=5,
                              inclusion_mode='any_vertex'):
    """
    Find triangles within a specified radius of a station using spherical distance.

    Parameters:
    -----------
    station_lat : float
        Latitude of the station
    station_lng : float
        Longitude of the station
    verts : DataFrame
        DataFrame with 'x' (longitude) and 'y' (latitude) columns
    tris : DataFrame
        DataFrame with 'v0', 'v1', 'v2' columns containing vertex indices
    radius_km : float
        Search radius in kilometers
    inclusion_mode : str
        How to determine if a triangle is included:
        - 'any_vertex': Include if any vertex is within radius
        - 'all_vertices': Include if all vertices are within radius
        - 'centroid': Include if triangle centroid is within radius
        - 'any_or_intersect': Include if any vertex is within radius or triangle intersects the circle

    Returns:
    --------
    extended_verts : DataFrame
        Vertices of the included triangles (reindexed)
    nearby_tris : DataFrame
        Included triangles with updated vertex indices
    """

    # Calculate accurate distances for all vertices
    distances = haversine_distance(
        station_lat, station_lng,
        verts['y'].values, verts['x'].values
    )

    # Create a mask for vertices within radius
    vertex_within_radius = distances <= radius_km

    if inclusion_mode == 'any_vertex':
        # Find triangles with at least one vertex within radius
        vertices_in_radius = np.where(vertex_within_radius)[0]
        if len(vertices_in_radius) == 0:
            return pd.DataFrame(columns=['x', 'y']), pd.DataFrame(columns=['v0', 'v1', 'v2'])

        triangle_mask = np.isin(tris[['v0', 'v1', 'v2']].values, vertices_in_radius).any(axis=1)

    elif inclusion_mode == 'all_vertices':
        # Find triangles with all vertices within radius
        tri_vertices = tris[['v0', 'v1', 'v2']].values
        triangle_mask = np.all(vertex_within_radius[tri_vertices], axis=1)

    elif inclusion_mode == 'centroid':
        # Calculate triangle centroids and check if they're within radius
        tri_vertices = tris[['v0', 'v1', 'v2']].values

        # Extract coordinates for all triangles more efficiently
        v0_coords = verts.iloc[tri_vertices[:, 0]][['x', 'y']].values
        v1_coords = verts.iloc[tri_vertices[:, 1]][['x', 'y']].values
        v2_coords = verts.iloc[tri_vertices[:, 2]][['x', 'y']].values

        # Calculate centroids
        centroid_lngs = (v0_coords[:, 0] + v1_coords[:, 0] + v2_coords[:, 0]) / 3
        centroid_lats = (v0_coords[:, 1] + v1_coords[:, 1] + v2_coords[:, 1]) / 3

        centroid_distances = haversine_distance(
            station_lat, station_lng,
            centroid_lats, centroid_lngs
        )
        triangle_mask = centroid_distances <= radius_km

    elif inclusion_mode == 'any_or_intersect':
        # More complex: include if any vertex is within radius OR
        # if the triangle intersects with the search circle
        # This is the most accurate but also most computationally expensive

        vertices_in_radius = np.where(vertex_within_radius)[0]
        triangle_mask = np.isin(tris[['v0', 'v1', 'v2']].values, vertices_in_radius).any(axis=1)

        # Additionally check triangles whose vertices are all outside but might intersect
        # This requires checking if the circle intersects with triangle edges
        # For simplicity, we'll also include triangles whose centroids are within 1.5x radius
        tri_vertices = tris[['v0', 'v1', 'v2']].values
        potentially_intersecting = ~triangle_mask  # Triangles not already included

        if potentially_intersecting.any():
            # Use the same efficient method for centroids
            pi_tri_vertices = tri_vertices[potentially_intersecting]
            v0_coords = verts.iloc[pi_tri_vertices[:, 0]][['x', 'y']].values
            v1_coords = verts.iloc[pi_tri_vertices[:, 1]][['x', 'y']].values
            v2_coords = verts.iloc[pi_tri_vertices[:, 2]][['x', 'y']].values

            centroid_lngs = (v0_coords[:, 0] + v1_coords[:, 0] + v2_coords[:, 0]) / 3
            centroid_lats = (v0_coords[:, 1] + v1_coords[:, 1] + v2_coords[:, 1]) / 3

            centroid_distances = haversine_distance(
                station_lat, station_lng,
                centroid_lats, centroid_lngs
            )

            # Add triangles whose centroids are within 1.5x radius (approximation for intersection)
            additional_mask = centroid_distances <= radius_km * 1.5
            triangle_mask[potentially_intersecting] |= additional_mask

    else:
        raise ValueError(f"Unknown inclusion_mode: {inclusion_mode}")

    # Get the selected triangles
    nearby_tris = tris[triangle_mask].copy()

    if len(nearby_tris) == 0:
        return pd.DataFrame(columns=['x', 'y']), pd.DataFrame(columns=['v0', 'v1', 'v2'])

    # Get all vertices that are part of these triangles
    all_tri_verts = np.unique(nearby_tris[['v0', 'v1', 'v2']].values.flatten())
    extended_verts = verts.iloc[all_tri_verts].copy().reset_index()

    # Create new index mapping
    old_indices = extended_verts['index'].values
    new_indices = np.arange(len(extended_verts))
    index_map = dict(zip(old_indices, new_indices))

    # Remap triangle indices
    for col in ['v0', 'v1', 'v2']:
        nearby_tris[col] = nearby_tris[col].map(index_map)

    # Clean up dataframes
    extended_verts = extended_verts.drop('index', axis=1).reset_index(drop=True)
    nearby_tris = nearby_tris.reset_index(drop=True)

    return extended_verts, nearby_tris

In [ ]:
# Create a simplified Folium map showing mesh outlines only
import folium
from folium import plugins


def create_simplified_folium_map(texas_stations, verts, tris, radius_km=15):
    """
    Create a cleaner Folium map showing just mesh outlines and closest triangles to each station
    """
    # Create base map
    center_lat = texas_stations['lat'].mean()
    center_lng = texas_stations['lng'].mean()

    simple_map = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=7,
        tiles=None
    )

    # Add tile layers
    folium.TileLayer('OpenStreetMap', name='OpenStreetMap', overlay=False, control=True).add_to(simple_map)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Satellite', overlay=False, control=True
    ).add_to(simple_map)

    # Color palette
    # colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightblue',
    #           'pink', 'darkgreen', 'cadetblue', 'darkpurple', 'lightgreen']
    colors = ['red']

    # For each station, find the closest few triangles and highlight them
    triangles_added = 0
    for idx, station in texas_stations.iterrows():
        station_id = station['id']
        station_name = station['name']
        station_lat = station['lat']
        station_lng = station['lng']
        color = colors[idx % len(colors)]

        print(f"Processing station {station_id}: {station_name}")

        # Use the find_surrounding_triangles function with 'centroid' mode for better accuracy
        local_verts, local_tris = find_surrounding_triangles(
            station_lat, station_lng, verts, tris, radius_km, inclusion_mode='centroid'
        )

        print(f"  Found {len(local_verts)} vertices, {len(local_tris)} triangles")

        if len(local_verts) == 0 or len(local_tris) == 0:
            print(f"  No mesh data found for station {station_id}")
            continue

        # Find the closest triangles to the station using accurate haversine distance
        closest_triangles = []
        for tri_idx, triangle in local_tris.iterrows():
            v0, v1, v2 = triangle['v0'], triangle['v1'], triangle['v2']

            # Calculate triangle center
            tri_center_lat = (local_verts.iloc[v0]['y'] + local_verts.iloc[v1]['y'] + local_verts.iloc[v2]['y']) / 3
            tri_center_lng = (local_verts.iloc[v0]['x'] + local_verts.iloc[v1]['x'] + local_verts.iloc[v2]['x']) / 3

            # Use haversine distance for accurate calculation
            distance = haversine_distance(station_lat, station_lng, tri_center_lat, tri_center_lng)
            closest_triangles.append((distance, triangle, tri_idx))

        # Sort by distance and take closest 10
        closest_triangles.sort(key=lambda x: x[0])

        # Add the 10 closest triangles
        show_triangles = min(10, len(closest_triangles))
        for i, (distance, triangle, tri_idx) in enumerate(closest_triangles[:show_triangles]):
            v0, v1, v2 = triangle['v0'], triangle['v1'], triangle['v2']

            # Verify indices are within bounds
            if v0 >= len(local_verts) or v1 >= len(local_verts) or v2 >= len(local_verts):
                print(f"  Warning: Invalid vertex indices for triangle {tri_idx}")
                continue

            triangle_coords = [
                [local_verts.iloc[v0]['y'], local_verts.iloc[v0]['x']],
                [local_verts.iloc[v1]['y'], local_verts.iloc[v1]['x']],
                [local_verts.iloc[v2]['y'], local_verts.iloc[v2]['x']]
            ]

            # Check if coordinates are valid
            if any(coord[0] < -90 or coord[0] > 90 or coord[1] < -180 or coord[1] > 180 for coord in triangle_coords):
                print(f"  Warning: Invalid coordinates for triangle {tri_idx}")
                continue

            depths = [local_verts.iloc[v0]['z'], local_verts.iloc[v1]['z'], local_verts.iloc[v2]['z']]
            avg_depth = np.mean(depths)

            # More visible styling
            # fill_opacity = 0.4 - (i * 0.05)  # Closest triangle most opaque
            fill_opacity = 0.4
            line_opacity = 0.9

            folium.Polygon(
                locations=triangle_coords,
                color='navy',
                weight=2,  # Thicker lines
                opacity=line_opacity,
                fillColor='lightblue',
                fillOpacity=fill_opacity,
                popup=f"<b>Station {station_id}</b><br>{station_name}<br>Triangle #{i+1} (rank {i+1})<br>Avg Depth: {avg_depth:.1f}m<br>Distance: {distance:.2f}km"
            ).add_to(simple_map)

            triangles_added += 1
            # print(f"  Added triangle {i+1} for station {station_id}")

    print(f"Total triangles added to map: {triangles_added}")

    # Add station markers
    for idx, station in texas_stations.iterrows():
        station_id = station['id']
        station_name = station['name']
        station_lat = station['lat']
        station_lng = station['lng']
        color = colors[idx % len(colors)]

        # Get mesh stats using find_surrounding_triangles
        local_verts, local_tris = find_surrounding_triangles(
            station_lat, station_lng, verts, tris, radius_km, inclusion_mode='any_vertex'
        )

        popup_html = f"""
        <div style='font-size: 14px;'>
            <b>Station {station_id}</b><br>
            <b>{station_name}</b><br>
            Location: {station_lat:.4f}°N, {station_lng:.4f}°W<br>
            Nearby Mesh Elements: {len(local_tris)}<br>
            Showing: {show_triangles} closest triangles
        </div>
        """

        folium.CircleMarker(
            location=[station_lat, station_lng],
            radius=10,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"Station {station_id}: {station_name}",
            color='white',
            fillColor=color,
            fillOpacity=1.0,
            weight=3
        ).add_to(simple_map)

    folium.LayerControl().add_to(simple_map)

    # Simplified legend
    legend_html = f'''
    <div style="position: fixed;
                bottom: 50px; left: 50px; width: 250px; height: auto;
                background-color: white; border:2px solid grey; z-index:9999;
                font-size:12px; padding: 10px; border-radius: 5px;">
    <p><b>Texas Stations - Closest Mesh Elements</b></p>
    <p><i class="fa fa-circle" style="color:red"></i> Stations</p>
    <p><i class="fa fa-square" style="color:lightblue; border: 1px solid blue;"></i> {show_triangles} Closest Mesh Triangles per Station</p>
    </div>
    '''
    simple_map.get_root().html.add_child(folium.Element(legend_html))

    return simple_map

print(f"Creating map...")
simplified_map = create_simplified_folium_map(texas_stations, verts, tris, radius_km=15)

# Create subdirectory for maps
map_dir = "maps"
os.makedirs(map_dir, exist_ok=True)

# Save the simplified map
simplified_map.save(f"{map_dir}/texas_stations_mesh_map.html")
print(f"Saved simplified map as '{map_dir}/texas_stations_mesh_map.html'")


# Display the simplified map
simplified_map

In [ ]:
def create_station_map(station_row, verts, tris, radius_km=10):
    """
    Create a map focused on a single station and its surrounding mesh
    """
    station_lat = station_row['lat']
    station_lng = station_row['lng']
    station_name = station_row['name']
    station_id = station_row['id']

    # Get surrounding triangles
    local_verts, local_tris = find_surrounding_triangles(
        station_lat, station_lng, verts, tris, radius_km
    )

    if len(local_verts) == 0 or len(local_tris) == 0:
        print(f"No mesh data found near station {station_id}: {station_name}")
        return None

    # Create station point
    station_point = hv.Points(
        [(station_lng, station_lat, station_name)],
        kdims=['lng', 'lat'],
        vdims=['name']
    ).opts(
        size=15,
        color='red',
        marker='circle',
        line_color='white',
        line_width=2,
        alpha=1.0
    )

    # Create wireframe for local mesh
    local_wireframe = datashade(
        hv.TriMesh((local_tris, local_verts)).edgepaths,
        alpha=128
    )

    # Create the local trimesh with depth values
    local_trimesh = hv.TriMesh((local_tris, hv.Points(local_verts, vdims='z')))
    local_dmesh = dshade.rasterize(local_trimesh).opts(
        cmap=cmocean.cm.deep,
        colorbar=True,
        width=800,
        height=600
    )

    # Create the combined plot
    plot = (local_dmesh * local_wireframe * station_point).opts(
        title=f"Station {station_id}: {station_name}\nDepth and Mesh Detail"
    )

    return plot

# Generate maps for each station
print("Creating individual station maps...")
station_maps = []

for idx, station in texas_stations.iterrows():
    print(f"Processing station {station['id']}: {station['name']}")
    station_map = create_station_map(station, verts, tris, radius_km=15)
    if station_map is not None:
        station_maps.append(station_map)

print(f"Created {len(station_maps)} station maps")

# Display the first few maps as examples
if len(station_maps) > 0:
    print("Displaying first 3 station maps:")
    # for i, station_map in enumerate(station_maps[:3]):
    for i, station_map in enumerate(station_maps):
        print(f"\nStation Map {i+1}:")
        display(station_map)